In [8]:
# Use kernal base (Python 3.13.5)
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
import os

print(f"Loading .env from: {find_dotenv()}")
load_dotenv()  # Load environment variables from .env file

Loading .env from: /Users/prernaroy/Library/CloudStorage/OneDrive-UNSW/Year 2/disso/dissertation/.env


True

In [5]:
# === Configuration ===
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # Load the API key from the .env file
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in your environment before running this notebook.")

client = OpenAI(api_key=OPENAI_API_KEY)
client = OpenAI()

In [ ]:
# # Test a basic API request
# response = client.responses.create(
#     model="gpt-5.4-mini-2026-03-17", #dated model for reproducibility purposes
#     input="Write a one-sentence bedtimme story about a dragon."
# )

# print(response.output_text)

Under the silver moon, a gentle dragon curled around his tiny forest friends, warming them with his soft golden breath until they all drifted happily to sleep.


In [ ]:
# Use a file URL as an input - example from OpenAI documentation
response = client.responses.create(
    model="gpt-5.4-mini-2026-03-17",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Analyze the letter and provide a summary of the key points.",
                },
                {
                    "type": "input_file",
                    "file_url": "https://www.berkshirehathaway.com/letters/2024ltr.pdf",
                },
            ],
        },
    ]
)

print(response.output_text)

Here’s a concise summary of the key points in Warren Buffett’s Berkshire Hathaway letter:

### Main themes
- Buffett says Berkshire’s annual report should do more than provide required financial data; it should also explain how Berkshire thinks and what shareholders own.
- He emphasizes honesty about mistakes, saying Berkshire openly acknowledges errors in capital allocation and management selection rather than hiding them.

### Leadership and succession
- Buffett notes that Greg Abel will eventually replace him as CEO and continue Berkshire’s tradition of candid reporting.
- He warns against fooling shareholders, saying that eventually you fool yourself too.

### Pete Liegl / Forest River story
- Buffett highlights Pete Liegl, founder of Forest River, as an exceptional manager and negotiator.
- He uses Liegl’s story to illustrate:
  - the value of integrity and talent,
  - the importance of finding great managers,
  - and that even a single outstanding decision can create enormous lon

In [ ]:
# Example of structured data extraction using Pydantic models
class ResearchPaperExtraction(BaseModel):
    title: str
    authors: list[str]
    abstract: str
    keywords: list[str]

response = client.responses.parse(
    model="gpt-5.4-mini-2026-03-17",
    input=[
        {
            "role": "system",
            "content": "You are an expert at structured data extraction. You will be given unstructured text from a research paper and should convert it into the given structure.",
        },
        {
            "role": "user", "content": "..."},
    ],
    text_format=ResearchPaperExtraction,
)

research_paper = response.output_parsed

In [ ]:
# Define the structured schema we want the model to return.
class ResearchPaperExtraction(BaseModel):
    title: str
    authors: list[str]
    abstract: str
    keywords: list[str]

client = OpenAI()

# Define the parsing request to extract structured data from a research paper.
response = client.responses.parse(
    model="gpt-5.4-mini-2026-03-17",
    input=[
        {
            "role": "system",
            "content": "You are an expert at structured data extraction. You will be given unstructured text from a research paper and should convert it into the given structure.",
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Please extract title, authors, abstract, and keywords from the linked paper."
                },
                {
                    "type": "input_file",
                    "file_url": "https://arxiv.org/pdf/2303.08774.pdf",
                }
            ]
        }

    ],
    text_format=ResearchPaperExtraction,
)

research_paper = response.output_parsed
print(research_paper)

title='GPT-4 Technical Report' authors=['OpenAI'] abstract='We report the development of GPT-4, a large-scale, multimodal model which can accept image and text inputs and produce text outputs. While less capable than humans in many real-world scenarios, GPT-4 exhibits human-level performance on various professional and academic benchmarks, including passing a simulated bar exam with a score around the top 10% of test takers. GPT-4 is a Transformer-based model pre-trained to predict the next token in a document. The post-training alignment process results in improved performance on measures of factuality and adherence to desired behavior. A core component of this project was developing infrastructure and optimization methods that behave predictably across a wide range of scales. This allowed us to accurately predict some aspects of GPT-4’s performance based on models trained with no more than 1/1,000th the compute of GPT-4.' keywords=['GPT-4', 'large-scale multimodal model', 'Transforme

Following the success of this method, I will now use this to test with a Coroner's report with the following variables:
1. Mental health condition flag 
2. Mental health label 
3. Developmental disability flag
4. Developmental disability label